# 🎵 Обучение Custom Demucs на MUSDB18 — полный режим

Ноутбук сам определяет, какой датасет доступен, готовит его и обучает модель.

| Режим | Что это | Качество |
|---|---|---|
| **full** | полный MUSDB18: 100 треков по ~4 минуты, лежит на Google Drive | настоящее — в 30 раз больше звука, чем в sample |
| **sample** | 7-секундные выдержки MUSDB18: 100 треков по 7 секунд | демо — общий принцип поймёт, для продакшена мало |

**Что происходит:** поиск данных → конвертация в формат проекта → (для маленьких
данных) лёгкое разнообразивание → 30 эпох обучения → сохранение модели на Drive.

## Про время и «неожиданную» скорость

30 эпох на Tesla T4 — ориентировочно **15–25 минут**. Да, быстро, и вот почему:
за одну эпоху сеть берёт **один случайный кусок 7.8 секунды из каждого трека**,
а не всю песню целиком. Полная 4-минутная дорожка и 7-секундная обрезка стоят
в эпоху одинаково. Обратная сторона: за эпоху сеть видит лишь малую часть
каждой длинной дорожки, поэтому 30 эпох — это «быстрый прогон», а не полная
проработка датасета.

⚠️ **Не жми «Run all» вслепую.** Сначала дождись, пока конвертация напечатает
число найденных треков, и убедись, что оно не ноль.

In [ ]:
# Подключаем Google Drive — оттуда берём полный MUSDB18 и туда же сохраняем модель.
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# Клонируем репозиторий с моделью и обучающим кодом.
import os
import shutil

REPO = "https://github.com/IgorCim/demucs_custom.git"
PROJECT_DIR = "/content/demucs_custom"

if os.path.isdir(PROJECT_DIR):
    print("Репозиторий уже скачан, обновляю.")
    shutil.rmtree(PROJECT_DIR)

!git clone --depth 1 {REPO}
os.chdir(PROJECT_DIR)
print("Работаем в:", os.getcwd())

In [ ]:
# Зависимости. demucs ставится из PyPI, но локальная папка репозитория
# перекрывает его — используется наша модифицированная версия.
#
# musdb ставится отдельно: в requirements.txt его нет, а без него
# не скачаются 7-секундные выдержки.
!pip install -q -r requirements.txt
!pip install -q musdb

# ffmpeg ОБЯЗАТЕЛЕН для 7-секундного сэмпла: скачивается не набор wav,
# а плоский список .stem.mp4, где каждый файл — контейнер с 5 аудиопотоками.
# Распаковать его без ffmpeg нечем, и конвертация упадёт с нулём треков.
import subprocess
import shutil
if shutil.which("ffmpeg") is None:
    print("⬇️  Ставлю ffmpeg...")
    subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=False)
ffmpeg_ok = shutil.which("ffmpeg") is not None
print("ffmpeg   :", "✅ есть" if ffmpeg_ok else "❌ нет")
if not ffmpeg_ok:
    raise RuntimeError("ffmpeg не найден — ячейка 8 не сможет распаковать .stem.mp4")

# sanity check
import importlib
for name in ("torch", "numpy", "soundfile", "librosa", "musdb", "demucs"):
    importlib.import_module(name)
print("Все зависимости на месте.")

In [ ]:
# Проверяем GPU до обучения — на CPU 30 эпох считать бессмысленно.
import torch

print("torch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Видеопамять: {total_gb:.1f} ГБ")
else:
    raise RuntimeError(
        "GPU не найдена. В Colab: Среда выполнения -> Сменить среду выполнения "
        "-> Аппаратный ускоритель -> GPU (T4)."
    )

In [ ]:
# Все пути и константы обучения.
#
# ВАЖНО про длину куска: 343980 отсчётов = 7.8 секунды. Это ЖЁСТКИЙ ПРЕДЕЛ
# pretrained htdemucs, а не наш выбор. Его segment = 39/5 секунды, и сеть
# отказывается обрабатывать что-то длиннее. Проверено на demucs 4.1.0:
#   343980 -> ок
#   480000 -> RuntimeError: shape '[1, 4, -1, 343980]' is invalid
#   485100 (44100*11) -> ValueError про training length
# Значение 480000 (10.88 сек) — это длина сегмента в НЕ-hybrid demucs,
# у htdemucs она своя. Поэтому здесь НЕ 11 секунд.

SEGMENT_LENGTH = 343980          # отсчётов, 7.8 сек — предел htdemucs
TARGET_SR = 44100                # частота дискретизации
EPOCHS = 30
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
VALID_RATIO = 0.1                # 10% треков уйдёт в валидацию

# Готовим данные на ЛОКАЛЬНОМ диске Colab — не на Drive.
# С Drive идёт постоянное чтение, оно убивает скорость обучения в разы.
# С Drive мы только ЧИТАЕМ исходный датасет (один раз) и ПИШЕМ готовую модель.
#
# Имя папки отличается от colab_training_v2, который пишет в /content/custom_dataset:
# оба ноутбука монтируются в один и тот же Colab, и общая папка означала бы,
# что второй ноутбук посчитает данные готовыми и пропустит подготовку.
CONVERTED_ROOT = "/content/custom_dataset_full"

# Длина куска подбирается под данные, см. ячейку 7:
#   full   -> 343980 (7.8 сек, предел htdemucs)
#   sample -> 264600 (6.0 сек, треки в сэмпле по 6.8 сек)
# Пока ставим максимум для полного датасета, ячейка 7 уточнит.
SEGMENT_LENGTH = 343980

OUTPUT_MODEL_PATH = (
    "/content/drive/MyDrive/demucs_custom/models/custom_demucs_final.pth"
)

print(f"Кусок:      {SEGMENT_LENGTH} отсчётов = {SEGMENT_LENGTH / TARGET_SR:.1f} сек")
print(f"Данные:     {CONVERTED_ROOT}")
print(f"Модель:     {OUTPUT_MODEL_PATH}")

In [ ]:
# Ячейка 7 — автоопределение источника данных.
#
# Приоритет:
#   1. полный MUSDB18 (больше 50 треков в train) на Drive или на локальном диске
#   2. 7-секундные выдержки MUSDB18
#   3. автоматическая загрузка выдержек
import os

SOURCES = ["vocals", "drums", "bass", "other"]


def find_best_dataset():
    full_musdb_paths = [
        '/content/drive/MyDrive/demucs_custom/data/musdb18_full',
        '/content/musdb18_full',
    ]

    for path in full_musdb_paths:
        train = os.path.join(path, "train")
        # больше 50 треков = это точно полный датасет, а не выдержки
        if os.path.isdir(train) and len(os.listdir(train)) > 50:
            print(f"✅ Полный MUSDB18: {path}")
            return path, 'full'

    sample_path = '/content/musdb18'
    if os.path.isdir(os.path.join(sample_path, "train")):
        print(f"⚠️  7-сек выдержки уже скачаны: {sample_path}")
        return sample_path, 'sample'

    print("⬇️  Скачиваю 7-сек выдержки MUSDB18 (~1 ГБ, 1-2 минуты)...")
    import musdb
    musdb.DB(root=sample_path, download=True)

    # musdb иногда отдаёт пустую папку при обрыве связи — проверяем,
    # иначе обучение молча ушло бы в режим "демо" на пустом месте.
    train = os.path.join(sample_path, "train")
    if not os.path.isdir(train) or not os.listdir(train):
        raise RuntimeError(
            f"Скачалось пусто: {sample_path}\n"
            "Проверь соединение и запусти ячейку заново."
        )

    print(f"✅ Выдержки готовы: {sample_path}")
    return sample_path, 'sample'


data_path, data_type = find_best_dataset()

# Длина куска под данные. У сэмпла треки по 6.8 секунды, и CustomDataset
# добивает короткий трек НУЛЯМИ до segment_length. При 343980 в каждом
# батче было бы около секунды тишины — модель бы обучалась на паузах.
# 264600 = 6.0 сек, столько влезает в сэмпл и не превышает предел htdemucs.
if data_type == 'sample':
    SEGMENT_LENGTH = 264600
    print(f"\nДлина куска: {SEGMENT_LENGTH} отсчётов = "
          f"{SEGMENT_LENGTH / TARGET_SR:.1f} сек (под треки по 6.8 сек)")
else:
    SEGMENT_LENGTH = 343980
    print(f"\nДлина куска: {SEGMENT_LENGTH} отсчётов = "
          f"{SEGMENT_LENGTH / TARGET_SR:.1f} сек (предел htdemucs)")

# Считаем, что реально доступно для обучения.
raw_train_dir = os.path.join(data_path, "train")
if not os.path.isdir(raw_train_dir):
    raw_train_dir = data_path
n_available = 0
if os.path.isdir(raw_train_dir):
    n_available = len(os.listdir(raw_train_dir))

if data_type == 'full':
    print(f"\nРежим: FULL   | треков в train: {n_available} | полные песни ~4 мин")
    print("Ожидаем: 15-25 минут на 30 эпох, качество — настоящее.")
else:
    print(f"\nРежим: SAMPLE | треков в train: {n_available} | по 7 секунд")
    print("Ожидаем: 10-15 минут на 30 эпох, качество — демонстрационное.")
    print("Для настоящего качества загрузи полный MUSDB18 (инструкция в конце).")

In [ ]:
# Ячейка 8 — подготовка данных в формат проекта.
#
# СКАЧАННЫЙ MUSDB ВЫГЛЯДИТ ПО-РАЗНОМУ В ЗАВИСИМОСТИ ОТ РЕЖИМА, и оба
# варианта нужно уметь обработать:
#
#   1) sample (7 секунд) — ПЛОСКИЙ список файлов:
#        /content/musdb18/train/Название трека.stem.mp4
#      Каждый .stem.mp4 — это НЕ аудиофайл, а контейнер сразу с 5 аудиопотоками
#      (AAC). Распаковывать его умеет только ffmpeg. Папок с дорожками
#      внутри нет вообще, поэтому искать подпапки бессмысленно.
#
#   2) full (с Zenodo) — обычные папки с wav:
#        /content/musdb18_full/train/Название трека/{vocals,drums,...}.wav
#      Папки train/, test/ и valid/ там есть, mixture.wav может не быть.
#
# Дальше оба варианта сводятся к общему шагу: получить 5 wav-файлов
# (mixture + 4 дорожки) на каждый трек и разложить их по train/valid.
#
# Про порядок потоков внутри .stem.mp4 — он задан в mus.yaml у demucs:
#     0 = mixture, 1 = drums, 2 = bass, 3 = other, 4 = vocals
import glob
import shutil
import subprocess
import time

import numpy as np
import soundfile as sf

print("=" * 70)
print("СТРУКТУРА ПОСЛЕ СКАЧИВАНИЯ")
print("=" * 70)
!ls -la {data_path} | head -20

STREAM_ORDER = ["mixture", "drums", "bass", "other", "vocals"]
SOURCES = ["vocals", "drums", "bass", "other"]

MAX_SAMPLES = 10 * 60 * TARGET_SR      # длиннее 10 минут не держим
# Сэмплы по 6.8 секунды, поэтому 10-секундный фильтр отсёк бы вообще всё.
MIN_SECONDS = 4.0 if data_type == 'sample' else 10.0
MIN_SAMPLES = int(MIN_SECONDS * TARGET_SR)


def read_stereo(path, max_samples):
    """Читает файл, приводит к 2 каналам 44100 Гц, режет по длине."""
    audio, sr = sf.read(path, dtype="float32", always_2d=True)
    if audio.shape[1] == 1:
        audio = np.repeat(audio, 2, axis=1)       # моно -> стерео
    elif audio.shape[1] > 2:
        audio = audio[:, :2]                      # лишние каналы отбрасываем
    if sr != TARGET_SR:
        import librosa
        audio = librosa.resample(
            audio.T, orig_sr=sr, target_sr=TARGET_SR
        ).T
    if audio.shape[1] > max_samples:
        audio = audio[:, :max_samples]
    return np.ascontiguousarray(audio.T, dtype="float32")   # -> (2, T)


def write_wav(path, audio, gain=1.0):
    """16-битный PCM: вдвое меньше места, чем float32.

    gain один и тот же на все пять файлов трека: пик суммы четырёх дорожек
    почти всегда больше 1.0, и при посегментной нормировке mixture уехал бы
    вниз относительно дорожек.
    """
    sf.write(path, (audio * gain).T, TARGET_SR, subtype="PCM_16")


def extract_stems(mp4_path, out_dir):
    """Распаковывает 5 потоков из .stem.mp4 в отдельные wav через ffmpeg."""
    os.makedirs(out_dir, exist_ok=True)
    for index, name in enumerate(STREAM_ORDER):
        out_path = os.path.join(out_dir, f"{name}.wav")
        # -map 0:a:N обязателен: wav-муксер не умеет писать несколько
        # аудиодорожек в один файл, без него получится каша.
        result = subprocess.run(
            ["ffmpeg", "-y", "-v", "error", "-i", mp4_path,
             "-map", f"0:a:{index}", "-ac", "2", "-ar", str(TARGET_SR),
             "-c:a", "pcm_s16le", out_path],
            capture_output=True, text=True, encoding="utf-8", errors="replace",
        )
        if result.returncode != 0 or not os.path.isfile(out_path):
            return False
    return True


# ---------------------------------------------------------------------------
# Шаг 1. Выясняем, что именно скачалось
# ---------------------------------------------------------------------------
train_root = os.path.join(data_path, "train")
if not os.path.isdir(train_root):
    train_root = data_path

mp4_files = sorted(
    glob.glob(os.path.join(train_root, "**", "*.stem.mp4"), recursive=True)
)
# test/ — официальный отладочный набор, на нём не обучаемся
mp4_files = [p for p in mp4_files if f"{os.sep}test{os.sep}" not in p]

folders = sorted(
    os.path.join(train_root, d)
    for d in os.listdir(train_root)
    if os.path.isdir(os.path.join(train_root, d))
) if os.path.isdir(train_root) else []

print("=" * 70)
if mp4_files:
    LAYOUT = "mp4"
    print(f"Найдено .stem.mp4 (плоские файлы): {len(mp4_files)}")
    print("Примеры:")
    for path in mp4_files[:3]:
        print("   ", os.path.basename(path))
    print("Нужно распаковать ffmpeg-ом: внутри каждого 5 аудиопотоков.")
elif folders:
    LAYOUT = "folders"
    print(f"Найдено папок с дорожками: {len(folders)}")
    print("Распаковка не нужна, читаем wav напрямую.")
else:
    raise RuntimeError(
        f"Не нашёл ни .stem.mp4, ни папок с дорожками в {train_root}\n"
        "Что скачалось — видно в распечатке выше."
    )

# ---------------------------------------------------------------------------
# Шаг 2. Приводим к единому виду: папка с 5 wav на каждый трек
# ---------------------------------------------------------------------------
if os.path.isdir(os.path.join(CONVERTED_ROOT, "train")) and os.path.isdir(
    os.path.join(CONVERTED_ROOT, "valid")
):
    print("\n✅ Датасет уже подготовлен, пропускаем.")
    STAGE_DIR = None
else:
    STAGE_DIR = os.path.join(CONVERTED_ROOT, "_staging")
    if os.path.isdir(STAGE_DIR):
        shutil.rmtree(STAGE_DIR)       # прерванный прошлый запуск
    os.makedirs(STAGE_DIR, exist_ok=True)

    if LAYOUT == "mp4":
        if shutil.which("ffmpeg") is None:
            raise RuntimeError(
                "ffmpeg не найден, а .stem.mp4 без него не распаковать.\n"
                "Ячейка 3 должна ставить ffmpeg. Перезапусти её."
            )
        print(f"\nРаспаковываю {len(mp4_files)} треков из .stem.mp4 ...")
        started = time.time()
        for index, mp4_path in enumerate(mp4_files):
            out_dir = os.path.join(STAGE_DIR, f"track_{index + 1:03d}")
            if not extract_stems(mp4_path, out_dir):
                print(f"  ⚠️  не распаковался: {os.path.basename(mp4_path)}")
                shutil.rmtree(out_dir, ignore_errors=True)
            if (index + 1) % 20 == 0:
                print(f"  ... {index + 1} из {len(mp4_files)}")
        print(f"✅ Распаковано за {time.time() - started:.0f} сек")
    else:
        for index, src in enumerate(folders):
            shutil.copytree(
                src, os.path.join(STAGE_DIR, f"track_{index + 1:03d}")
            )

# ---------------------------------------------------------------------------
# Шаг 3. Отбор, разбиение на train/valid, запись
# ---------------------------------------------------------------------------
if STAGE_DIR is not None:
    if os.path.isdir(os.path.join(CONVERTED_ROOT, "train")):
        shutil.rmtree(os.path.join(CONVERTED_ROOT, "train"))
    if os.path.isdir(os.path.join(CONVERTED_ROOT, "valid")):
        shutil.rmtree(os.path.join(CONVERTED_ROOT, "valid"))
    os.makedirs(os.path.join(CONVERTED_ROOT, "train"), exist_ok=True)
    os.makedirs(os.path.join(CONVERTED_ROOT, "valid"), exist_ok=True)

    candidates = sorted(os.listdir(STAGE_DIR))

    # Отбор ДО разбиения: часть треков может не распаковаться, и если
    # резать сначала, валидация рискует остаться пустой.
    # mixture.wav не обязателен: у полного MUSDB он обычно есть, а если
    # нет — соберём его из дорожек на следующем шаге.
    usable = []
    for name in candidates:
        src_dir = os.path.join(STAGE_DIR, name)
        if all(
            os.path.isfile(os.path.join(src_dir, f"{s}.wav"))
            for s in SOURCES
        ):
            usable.append(name)
        else:
            print(f"  ⚠️  пропускаю {name}: нет части дорожек")

    if not usable:
        raise RuntimeError(
            "Ни один трек не прошёл проверку. Смотри сообщения выше — "
            "ffmpeg не смог распаковать .stem.mp4."
        )

    n_valid = max(1, int(len(usable) * VALID_RATIO))
    started = time.time()
    written = 0

    for index, name in enumerate(usable):
        src_dir = os.path.join(STAGE_DIR, name)
        mix_path = os.path.join(src_dir, "mixture.wav")
        have_mixture = os.path.isfile(mix_path)
        try:
            mixture = (
                read_stereo(mix_path, MAX_SAMPLES) if have_mixture else None
            )
            stems = [
                read_stereo(os.path.join(src_dir, f"{s}.wav"), MAX_SAMPLES)
                for s in SOURCES
            ]
        except Exception as exc:
            print(f"  ⚠️  не читается {name}: {type(exc).__name__}")
            continue

        # Дорожки могут отличаться по длине — обрезаем все до самой короткой,
        # иначе сложение невозможно.
        length = min(
            [stems[0].shape[1]]
            + ([mixture.shape[1]] if have_mixture else [])
            + [s.shape[1] for s in stems]
        )
        if length < MIN_SAMPLES:
            print(f"  ⚠️  пропускаю {name}: всего {length / TARGET_SR:.1f} сек")
            continue
        if have_mixture:
            mixture = mixture[:, :length]
        stems = [s[:, :length] for s in stems]

        if LAYOUT == "mp4" or have_mixture:
            # Готовая смесь: из .stem.mp4 это поток 0, в полном MUSDB -
            # официальный mixture.wav. Ей верим и НЕ пересобираем.
            # Внутри AAC есть артефакты, поэтому mixture не равна сумме
            # дорожек идеально — так и в настоящем датасете.
            gain = 1.0
        else:
            # mixture.wav не было — собираем смесь из дорожек.
            # Нормируем все пять файлов одним коэффициентом, иначе сумма
            # дорожек уедет выше 1.0 и mixture от неё отстанет.
            mixture = np.sum(stems, axis=0)
            peak = float(np.max(np.abs(mixture)))
            gain = (1.0 / peak) if peak > 1.0 else 1.0

        is_valid = index >= len(usable) - n_valid
        split = "valid" if is_valid else "train"
        out_dir = os.path.join(CONVERTED_ROOT, split, name)
        os.makedirs(out_dir, exist_ok=True)

        write_wav(os.path.join(out_dir, "mixture.wav"), mixture, gain)
        for s_index, source_name in enumerate(SOURCES):
            write_wav(
                os.path.join(out_dir, f"{source_name}.wav"), stems[s_index], gain
            )
        written += 1

    shutil.rmtree(STAGE_DIR, ignore_errors=True)
    print(f"\n✅ Обработано {written} треков за {time.time() - started:.0f} сек")

    # -------------------------------------------------------------------------
    # Шаг 4. Итог и проверка
    # -------------------------------------------------------------------------
    n_train = len([
        d for d in os.listdir(os.path.join(CONVERTED_ROOT, "train"))
        if os.path.isdir(os.path.join(CONVERTED_ROOT, "train", d))
    ])
    n_valid_actual = len([
        d for d in os.listdir(os.path.join(CONVERTED_ROOT, "valid"))
        if os.path.isdir(os.path.join(CONVERTED_ROOT, "valid", d))
    ])

    print("\n" + "=" * 70)
    print(f"✅ Треков в train: {n_train}")
    print(f"✅ Треков в valid: {n_valid_actual}")
    print(f"   Папка: {CONVERTED_ROOT}")
    print("=" * 70)

    # Без mixture.wav обучение тихо уходит в демо-режим на 2 эпохи.
    if n_train == 0:
        raise RuntimeError("train пустой — обучение не запустится нормально.")
    print("   Проверка: mixture.wav на месте, обучение будет полноценным.")

In [ ]:
# Ячейка 9 — разнообразивание данных.
#
# Честно о том, что это даёт: НИЧЕГО нового по содержанию. Мы не добываем
# ни одной секунды нового звука — делаем вариации уже имеющихся 7-секундных
# обрезков. Это может немного помочь не переобучиться под ровно эти куски,
# но НЕ превращает 100×7 секунд в полноценный датасет. Если хочется качества —
# загружайте полный MUSDB18 (инструкция в конце ноутбука).
#
# Для полного датасета аугментация выключена: там и так 100 длинных дорожек,
# а копии только замедляют обучение.
#
# Что делаем (для каждого трека из train/):
#   * копии со сдвигом тональности — одинаковым для ВСЕХ дорожек трека
#   * лёгкий шум в СМЕСЬ, дорожки-цели остаются чистыми
#
# Почему смесь собирается ИЗ сдвинутых дорожек, а не сдвигается отдельно:
# pitch_shift нелинеен (внутри фазовый вокодер) — проверено: расхождение
# достигает 100% амплитуды. Если сдвинуть дорожки по отдельности, mixture
# перестанет быть их суммой, и обучение станет бессмысленным. Поэтому сначала
# сдвигаем 4 дорожки, затем складываем их и добавляем шум.
#
# valid/ НЕ трогаем: иначе метрика на валидации станет скакать
# и выбор лучшей модели будет ненадёжным.
#
# ВНИМАНИЕ: здесь НЕ используется torchaudio.load/save — в этой версии
# они требуют torchcodec и падают. Читаем и пишем через soundfile.
if data_type == 'full':
    print("Режим FULL — данные достаточно, аугментация не нужна.")
    print("Копии длинных дорожек только замедлили бы обучение.")
else:
    try:
        import librosa
    except ImportError as exc:
        # Датасет к этому моменту уже готов и корректен. Аугментация —
        # приятное дополнение, а не условие обучения. Не роняем прогон.
        print(f"⚠️  librosa не установился ({exc}), пропускаю аугментацию.")
        print("   Обучение запустится на исходных данных — это нормально.")
        librosa = None

    if librosa is not None:
        NUM_COPIES = 3
        SEMITONES = [-2.0, 1.5, 3.0]
        NOISE_LEVEL = 0.002          # тихий, чтобы не выбить цели из смеси

        train_dir = os.path.join(CONVERTED_ROOT, "train")
        tracks = sorted(
            d for d in os.listdir(train_dir)
            if os.path.isdir(os.path.join(train_dir, d))
        )
        print(f"Аугментация: {len(tracks)} треков × {NUM_COPIES} копии")

        rng = np.random.default_rng(42)      # фиксированный seed — повторяемо
        started = time.time()
        created = 0

        for track in tracks:
            src_dir = os.path.join(train_dir, track)

            # Сначала читаем и сдвигаем дорожки. Отдельной "дорожки-эталона"
            # у MUSDB нет — эталон ровно тот, что записан в mixture.wav.
            for copy_index, semitones in enumerate(SEMITONES[:NUM_COPIES]):
                dst_dir = os.path.join(train_dir, f"{track}_aug{copy_index + 1}")
                if os.path.isdir(dst_dir):
                    continue

                shifted = []
                ok = True
                for name in SOURCES:
                    path = os.path.join(src_dir, f"{name}.wav")
                    if not os.path.isfile(path):
                        ok = False
                        break
                    audio, sr = sf.read(path, dtype="float32", always_2d=True)
                    audio = audio.T                       # -> (2, T)
                    if abs(semitones) > 1e-6:
                        audio = np.stack([
                            librosa.effects.pitch_shift(
                                audio[ch], sr=sr, n_steps=semitones
                            )
                            for ch in range(audio.shape[0])
                        ])
                    shifted.append(np.ascontiguousarray(audio, dtype="float32"))

                if not ok:
                    continue

                # Смесь собираем ИЗ СДВИНУТЫХ дорожек, а не сдвигаем отдельно.
                # Иначе mixture перестала бы быть их суммой: pitch_shift
                # нелинеен (фазовый вокодер), и независимый сдвиг дорожек
                # давал бы расхождение до 100% амплитуды. Так эталон и вход
                # гарантированно согласованы.
                stems = np.stack(shifted, axis=0)       # (4, 2, T)
                mixture = stems.sum(axis=0)
                # шум только в смеси: сеть учится доставать чистые цели
                # из слегка грязной смеси, цели остаются чистыми
                mixture = mixture + rng.normal(
                    0.0, NOISE_LEVEL, mixture.shape
                ).astype("float32")

                os.makedirs(dst_dir, exist_ok=True)
                # тот же общий gain на все пять файлов, что и при конвертации
                peak = float(np.max(np.abs(mixture)))
                gain = (1.0 / peak) if peak > 1.0 else 1.0
                for index, name in enumerate(SOURCES):
                    write_wav(os.path.join(dst_dir, f"{name}.wav"), stems[index], gain)
                write_wav(os.path.join(dst_dir, "mixture.wav"), mixture, gain)
                created += 1

        total = len(tracks) + created
        print(f"✅ Создано копий: {created} за {time.time() - started:.0f} сек")
        print(f"   Треков в train теперь: {total}")
        print("   ⚠️  Это вариации тех же 7 секунд. Нового звука не появилось.")

In [ ]:
# Ячейка 10 — обучение.
#
# ВАЖНО: segment_length = 343980, а не 480000 и не 44100*11. Это предел
# pretrained htdemucs. С значением 480000 обучение падает с RuntimeError,
# с 485100 (11 сек) — с ValueError про training length.
import importlib
import sys

for name in ("config", "train_custom", "data_prep", "model_loader", "train_epoch"):
    sys.modules.pop(name, None)

from config import Config
from train_custom import train_custom_model

config = Config()

config.device = "cuda"
config.segment_length = SEGMENT_LENGTH      # 343980 — обязательно
config.batch_size = BATCH_SIZE
config.epochs = EPOCHS
config.learning_rate = LEARNING_RATE
config.custom_data_path = CONVERTED_ROOT
config.output_model_path = OUTPUT_MODEL_PATH

print("Параметры обучения:")
print(f"  режим данных      : {data_type}")
print(f"  устройство        : {config.device}")
print(f"  длина куска       : {config.segment_length} отсчётов "
      f"({config.segment_length / TARGET_SR:.1f} сек)")
print(f"  батч              : {config.batch_size}")
print(f"  эпох              : {config.epochs}")
print(f"  learning rate     : {config.learning_rate}")
print(f"  данные            : {config.custom_data_path}")
print(f"  модель            : {config.output_model_path}")

# Сколько реально будет итераций — чтобы понимать, чего ждать.
try:
    n_train = len([
        d for d in os.listdir(os.path.join(CONVERTED_ROOT, "train"))
        if os.path.isdir(os.path.join(CONVERTED_ROOT, "train", d))
    ])
    iters = (n_train // config.batch_size) * config.epochs
    print(f"  треков в train    : {n_train}")
    print(f"  итераций всего    : ~{iters}")
except Exception:
    pass

# epochs_explicit=True — просим обучение НЕ решать, что 30 эпох это слишком
# много, иначе возможен тихий срез до 2 эпох.
best_model_path = train_custom_model(config, assume_yes=True, epochs_explicit=True)

print("\n" + "=" * 70)
print("✅ ОБУЧЕНИЕ ЗАВЕРШЕНО")
print(f"Модель сохранена: {best_model_path}")
print("=" * 70)

## Как получить полный MUSDB18 (режим `full`)

Только `musdb.DB(download=True)` скачивает **7-секундные выдержки** — это
официальное поведение пакета. Полный набор лежит на Zenodo и требует
одобрения заявки, которую вручную рассматривают (обычно до суток).

### Что сделать

1. **Запроси доступ.** Открой страницу набора на Zenodo и подай заявку,
   согласившись с лицензией CC BY-NC-SA 4.0 на некоммерческое использование.
2. **Скачай** полный архив (около 10 ГБ) и **распакуй** так, чтобы получилось:

   ```
   musdb18_full/
     train/            <- 100 треков, из них ноутбук возьмёт 90 в train и 10 в valid
       Track Name/
         vocals.wav
         drums.wav
         bass.wav
         other.wav
     test/             <- 50 треков, НЕ используются для обучения
   ```

3. **Положи** папку `musdb18_full` в Google Drive сюда:

   ```
   MyDrive/demucs_custom/data/musdb18_full/
   ```

4. **Перезапусти** ноутбук с ячейки 7. Он сам найдёт полный датасет
   (проверяет, что треков больше 50) и переключится в режим `full`.
   Конвертацию делать заново не нужно — она пропустит уже готовую папку,
   так что при смене режима её стоит удалить:
   `!rm -rf /content/custom_dataset_full`

### Почему папка `test` не используется

Это официальный отладочный набор. Если обучиться на нём, метрика
качества перестанет что-либо значить.

## Если что-то пошло не так

| Симптом | Причина | Что делать |
|---|---|---|
| `RuntimeError: shape '[1, 4, -1, 343980]' is invalid` | где-то осталось `segment_length = 480000` | проверить значение в ячейке 10 |
| `ValueError: Given length ... is longer than training length 343980` | задан кусок длиннее 7.8 сек | вернуть `SEGMENT_LENGTH = 343980` |
| `torchcodec` / ошибки `torchaudio.load` | torchaudio 2.11 требует torchcodec | в ноутбуке звук читается через `soundfile`, это обойдено |
| обучение закончилось за 2 эпохи | не нашлось ни одного `mixture.wav` | сработала защита в ячейке 8 — прочитать её сообщение |
| `CUDA available: False` | в Colab не выбрана GPU | Среда выполнения → Аппаратный ускоритель → GPU (T4) |
| Colab отвалился по таймауту | бесплатный Colab обрывает долгие сессии | сохранить чекпойнты на Drive, запускать заново |